In [ ]:
# feature_engineering.ipynb

import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import joblib
import os

CLEAN_PATH = "../data/cleaned_logs/cleaned_logs.csv"
OUTPUT_PATH = "../data/cleaned_logs/features.csv"

df = pd.read_csv(CLEAN_PATH)

# Basic features
df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
df['failed'] = df['action'].str.contains("fail|invalid", case=False, na=False).astype(int)
df['success'] = df['action'].str.contains("accept", case=False, na=False).astype(int)

# One-hot encode log_type
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(df[['log_type']])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['log_type']))
df = pd.concat([df, encoded_df], axis=1)

# Save encoder
joblib.dump(ohe, "../models/ohe_encoder.pkl")

# Select numeric-only features
feature_cols = ['hour', 'failed', 'success'] + list(encoded_df.columns)
df_features = df[feature_cols]

df_features.to_csv(OUTPUT_PATH, index=False)
df_features.head()
